In [1]:
'''
Overview
this notebook is a working example of scraping ufc fight stats one at a time
the code is broken down into small steps that can be run in parts to view and verify results at each step
this is useful for testing and debugging the code as the ufc stats website may be update, breaking the code


scrape ufc fight stats
get all event details, name, url, date, location for all ufc events
for each event, get fight details all fights on card
parse each fight to get fight stats of both fighters

additional scraping of fighter's details, name, nickname, url
for each fighter, get their tale of the tape
'''

'\nOverview\nthis notebook is a working example of scraping ufc fight stats one at a time\nthe code is broken down into small steps that can be run in parts to view and verify results at each step\nthis is useful for testing and debugging the code as the ufc stats website may be update, breaking the code\n\n\nscrape ufc fight stats\nget all event details, name, url, date, location for all ufc events\nfor each event, get fight details all fights on card\nparse each fight to get fight stats of both fighters\n'

In [5]:
# imports
import pandas as pd
import numpy as np
import os
import re
import requests
from bs4 import BeautifulSoup
import itertools

# Change working directory out of tests
os.chdir('..')

exec(open('./global.py').read())

# Parse Event Details
Includes:
<br>
Event
<br>
URL
<br>
Date
<br>
Location
<br>

In [7]:
# define url to parse (just do the first page)
url = config['completed_events_all_url'].replace('all', '1')

# get soup
soup = get_soup(url)

In [10]:
# parse event details
event_details_df = parse_event_details(soup)

# show event details
display(event_details_df)

,EVENT,URL,DATE,LOCATION
0,UFC Fight Night: Ribas vs. Namajunas,http://ufcstats.com/event-details/79ff6545b0ab...,"March 23, 2024","Las Vegas, Nevada, USA"
1,UFC Fight Night: Tuivasa vs. Tybura,http://ufcstats.com/event-details/c398235fcaf8...,"March 16, 2024","Las Vegas, Nevada, USA"
2,UFC 299: O'Malley vs. Vera 2,http://ufcstats.com/event-details/a9df5ae20a97...,"March 09, 2024","Miami, Florida, USA"
3,UFC Fight Night: Rozenstruik vs. Gaziev,http://ufcstats.com/event-details/e4a9dbade7c7...,"March 02, 2024","Las Vegas, Nevada, USA"
4,UFC Fight Night: Moreno vs. Royval 2,http://ufcstats.com/event-details/902ab9197b83...,"February 24, 2024","Mexico City, Distrito Federal, Mexico"
5,UFC 298: Volkanovski vs. Topuria,http://ufcstats.com/event-details/dab0e6cb8c93...,"February 17, 2024","Anaheim, California, USA"
6,UFC Fight Night: Hermansson vs. Pyfer,http://ufcstats.com/event-details/eaea0fc7b765...,"February 10, 2024","Las Vegas, Nevada, USA"
7,UFC Fight Night: Dolidze vs. Imavov,http://ufcstats.com/event-details/cce79e827569...,"February 03, 2024","Las Vegas, Nevada, USA"
8,UFC 297: Strickland vs. Du Plessis,http://ufcstats.com/event-details/bd85ca0bb4f2...,"January 20, 2024","Toronto, Ontario, Canada"
9,UFC Fight Night: Ankalaev vs. Walker 2,http://ufcstats.com/event-details/010986ee359f...,"January 13, 2024","Las Vegas, Nevada, USA"


# Parse Fight Details
Includes:
<br>
Event
<br>
Bout
<br>
URL

In [16]:
# parse one event for fight details

# define url to parse
url = event_details_df['URL'].iloc[1]
print(url)

# get soup
soup = get_soup(url)

http://ufcstats.com/event-details/c398235fcaf8d71d


In [17]:
# parse fight links
fight_details_df = parse_fight_details(soup)

# show fight links
display(fight_details_df)

,EVENT,BOUT,URL
0,UFC Fight Night: Tuivasa vs. Tybura,Marcin Tybura vs. Tai Tuivasa,http://ufcstats.com/fight-details/8bc0c220f7dc...
1,UFC Fight Night: Tuivasa vs. Tybura,Bryan Battle vs. Ange Loosa,http://ufcstats.com/fight-details/a7fd9da657bb...
2,UFC Fight Night: Tuivasa vs. Tybura,Ovince Saint Preux vs. Kennedy Nzechukwu,http://ufcstats.com/fight-details/35f9beb94b4e...
3,UFC Fight Night: Tuivasa vs. Tybura,Christian Rodriguez vs. Isaac Dulgarian,http://ufcstats.com/fight-details/acc7392d48cc...
4,UFC Fight Night: Tuivasa vs. Tybura,Macy Chiasson vs. Pannie Kianzad,http://ufcstats.com/fight-details/2f414494f2b6...
5,UFC Fight Night: Tuivasa vs. Tybura,Gerald Meerschaert vs. Bryan Barberena,http://ufcstats.com/fight-details/569e0ba18701...
6,UFC Fight Night: Tuivasa vs. Tybura,Mike Davis vs. Natan Levy,http://ufcstats.com/fight-details/4f20f3ada148...
7,UFC Fight Night: Tuivasa vs. Tybura,Chelsea Chandler vs. Josiane Nunes,http://ufcstats.com/fight-details/711ec0545552...
8,UFC Fight Night: Tuivasa vs. Tybura,Jafel Filho vs. Ode Osbourne,http://ufcstats.com/fight-details/451df53c1318...
9,UFC Fight Night: Tuivasa vs. Tybura,Danny Silva vs. Josh Culibao,http://ufcstats.com/fight-details/c26f27667c23...


# Parse Fight Results and Stats

In [18]:
# define url to parse
# various types of fights
# url = 'http://ufcstats.com/fight-details/4b7ec02b39fc6f70' # one round finish
# url = 'http://ufcstats.com/fight-details/8b3b38167060b1d7' # three rounds decision
# url = 'http://ufcstats.com/fight-details/b22eab3aa1522f40' # three rounds finish
# url = 'http://ufcstats.com/fight-details/3109d1151f149aaf' # five rounds decision
# url = 'http://ufcstats.com/fight-details/d93c8c77e1091a16' # no stats
# url = 'http://ufcstats.com/fight-details/c63edd25d2201a46' # draw
url = 'http://ufcstats.com/fight-details/37cb7ce0f0b70640' # point deduction

# get soup
soup = get_soup(url)

### Parse Fight Results

Includes:
<br>
Event
<br>
Bout
<br>
Weightclass
<br>
Method
<br>
Round
<br>
Time
<br>
Time Format
<br>
Referee
<br>
Details
<br>

In [19]:
# parse fight results from soup
fight_results = parse_fight_results(soup)
# append fight url
fight_results.append('URL:'+url)

# show fight results
fight_results

['UFC 33: Victory in Vegas ',
 'Dave Menne ',
 'Gil Castillo ',
 'W',
 'L',
 'UFC Middleweight Title Bout',
 'Method: Decision - Unanimous ',
 'Round:5',
 'Time:5:00',
 'Time format:5 Rnd (5-5-5-5-5)',
 'Referee:John McCarthy',
 'Details:Point Deducted: Illegal Knee by MenneTony Weeks 45 - 49.Doug Crosby 42 - 49.Jeff Mullen 44 - 49.',
 'URL:http://ufcstats.com/fight-details/37cb7ce0f0b70640']

In [20]:
# organise fight results
fight_results_df = organise_fight_results(fight_results, config['fight_results_column_names'])

# show fight results
display(fight_results_df)

,EVENT,BOUT,OUTCOME,WEIGHTCLASS,METHOD,ROUND,TIME,TIME FORMAT,REFEREE,DETAILS,URL
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,W/L,UFC Middleweight Title Bout,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),John McCarthy,Point Deducted: Illegal Knee by MenneTony Week...,http://ufcstats.com/fight-details/37cb7ce0f0b7...


### Parse Fight Stats
Includes:
Event
<br>
Bout
<br>
Round
<br>
Fighter
<br>
Kd
<br>
Sig.Str.
<br>
Sig.Str. %
<br>
Total Str.
<br>
Td
<br>
Td %
<br>
Sub.Att
<br>
Rev.
<br>
Ctrl
<br>
Head
<br>
Body
<br>
Leg
<br>
Distance
<br>
Clinch
<br>
Ground
<br>

In [22]:
# parse full fight stats for both fighters
fighter_a_stats, fighter_b_stats = parse_fight_stats(soup)

# show fighter stats
print(fighter_a_stats[:20])

['Dave Menne', '0', '70 of 125', '56%', '180 of 241', '1 of 1', '100%', '4', '2', '7:55', 'Dave Menne', '0', '15 of 22', '68%', '45 of 52', '1 of 1', '100%', '1', '0', '2:02']


In [23]:
# organise stats extracted from soup
fighter_a_stats_clean = organise_fight_stats(fighter_a_stats)
fighter_b_stats_clean = organise_fight_stats(fighter_b_stats)

# show organised stats
print(fighter_a_stats_clean[:2])

[['Dave Menne', '0', '70 of 125', '56%', '180 of 241', '1 of 1', '100%', '4', '2', '7:55'], ['Dave Menne', '0', '15 of 22', '68%', '45 of 52', '1 of 1', '100%', '1', '0', '2:02']]


In [24]:
# convert list of fighter stats into a structured dataframe
fighter_a_stats_df = convert_fight_stats_to_df(fighter_a_stats_clean, config['totals_column_names'], config['significant_strikes_column_names'])
fighter_b_stats_df = convert_fight_stats_to_df(fighter_b_stats_clean, config['totals_column_names'], config['significant_strikes_column_names'])

# show stats df
display(fighter_a_stats_df)

,ROUND,FIGHTER,KD,SIG.STR.,SIG.STR. %,TOTAL STR.,TD,TD %,SUB.ATT,REV.,CTRL,HEAD,BODY,LEG,DISTANCE,CLINCH,GROUND
0,Round 1,Dave Menne,0,15 of 22,68%,45 of 52,1 of 1,100%,1,0,2:02,5 of 9,8 of 10,2 of 3,2 of 4,8 of 10,5 of 8
1,Round 2,Dave Menne,0,6 of 15,40%,34 of 45,0 of 0,---,2,1,3:07,3 of 12,2 of 2,1 of 1,2 of 5,1 of 2,3 of 8
2,Round 3,Dave Menne,0,14 of 29,48%,29 of 47,0 of 0,---,0,1,0:14,7 of 21,3 of 3,4 of 5,10 of 24,4 of 5,0 of 0
3,Round 4,Dave Menne,0,21 of 41,51%,24 of 44,0 of 0,---,1,0,0:07,13 of 31,3 of 3,5 of 7,7 of 21,13 of 19,1 of 1
4,Round 5,Dave Menne,0,14 of 18,77%,48 of 53,0 of 0,---,0,0,2:25,10 of 13,0 of 0,4 of 5,3 of 5,4 of 6,7 of 7


In [25]:
# combine fighter stats into one
fight_stats = combine_fighter_stats_dfs(fighter_a_stats_df, fighter_b_stats_df, soup)

# show fight stats
display(fight_stats)

,EVENT,BOUT,ROUND,FIGHTER,KD,SIG.STR.,SIG.STR. %,TOTAL STR.,TD,TD %,SUB.ATT,REV.,CTRL,HEAD,BODY,LEG,DISTANCE,CLINCH,GROUND
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 1,Dave Menne,0,15 of 22,68%,45 of 52,1 of 1,100%,1,0,2:02,5 of 9,8 of 10,2 of 3,2 of 4,8 of 10,5 of 8
1,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 2,Dave Menne,0,6 of 15,40%,34 of 45,0 of 0,---,2,1,3:07,3 of 12,2 of 2,1 of 1,2 of 5,1 of 2,3 of 8
2,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 3,Dave Menne,0,14 of 29,48%,29 of 47,0 of 0,---,0,1,0:14,7 of 21,3 of 3,4 of 5,10 of 24,4 of 5,0 of 0
3,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 4,Dave Menne,0,21 of 41,51%,24 of 44,0 of 0,---,1,0,0:07,13 of 31,3 of 3,5 of 7,7 of 21,13 of 19,1 of 1
4,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 5,Dave Menne,0,14 of 18,77%,48 of 53,0 of 0,---,0,0,2:25,10 of 13,0 of 0,4 of 5,3 of 5,4 of 6,7 of 7
0,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 1,Gil Castillo,0,0 of 0,---,22 of 22,1 of 4,25%,0,0,2:27,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0
1,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 2,Gil Castillo,0,0 of 0,---,15 of 15,3 of 5,60%,0,0,1:05,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0,0 of 0
2,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 3,Gil Castillo,0,4 of 11,36%,38 of 45,1 of 3,33%,0,0,2:47,2 of 9,2 of 2,0 of 0,4 of 11,0 of 0,0 of 0
3,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 4,Gil Castillo,0,4 of 12,33%,21 of 29,1 of 2,50%,0,0,3:11,3 of 10,1 of 2,0 of 0,3 of 11,1 of 1,0 of 0
4,UFC 33: Victory in Vegas,Dave Menne vs. Gil Castillo,Round 5,Gil Castillo,0,1 of 2,50%,14 of 15,1 of 5,20%,0,0,2:06,0 of 1,0 of 0,1 of 1,0 of 1,1 of 1,0 of 0


# Parse Fighter Details

In [26]:
# define url to parse
url = 'http://ufcstats.com/statistics/fighters?char=b&page=all' # 'a' last names
url = 'http://ufcstats.com/statistics/fighters?char=b&page=all' # 'b' last names

In [27]:
# get soup
soup = get_soup(url)

In [28]:
# parse fighter details
fighter_details_df = parse_fighter_details(soup, config['fighter_details_column_names'])

# show fighter details
display(fighter_details_df)

,FIRST,LAST,NICKNAME,URL
0,Niklas,Backstrom,,http://ufcstats.com/fighter-details/39cc64bf0a...
1,Seth,Baczynski,The Polish Pistola,http://ufcstats.com/fighter-details/fd5b6598a3...
2,Abdul Azeem,Badakhshi,The Afghan Lion,http://ufcstats.com/fighter-details/968764372c...
3,Ryan,Bader,Darth,http://ufcstats.com/fighter-details/c0ab242c40...
4,Izabela,Badurek,,http://ufcstats.com/fighter-details/4fcf6e0c4e...
...,...,...,...,...
297,JP,Buys,Young Savage,http://ufcstats.com/fighter-details/0b36224780...
298,Dennis,Buzukja,The Great,http://ufcstats.com/fighter-details/c4d039123e...
299,Charles,Byrd,Kid Dynamite,http://ufcstats.com/fighter-details/76ee3d666c...
300,Steve,Byrnes,The Sergeant,http://ufcstats.com/fighter-details/4361245697...


# Parse Fighter Tale Of The Tape

In [29]:
# define url to parse
url = 'http://ufcstats.com/fighter-details/93fe7332d16c6ad9' # '--' and '' present
url = 'http://ufcstats.com/fighter-details/b361180739bed4b0' # '--' present
url = 'http://ufcstats.com/fighter-details/1897b7b913736a7c' # no first name
url = 'http://ufcstats.com/fighter-details/d0f3959b4a9747e6' # all tott present

In [30]:
# get soup
soup = get_soup(url)

In [31]:
# parse fighter tale of the tape
fighter_tott = parse_fighter_tott(soup)

# show tale of the tape
fighter_tott

['Fighter:Jose Aldo',
 'Height:5\' 7"',
 'Weight:135 lbs.',
 'Reach:70"',
 'STANCE:Orthodox',
 'DOB:Sep 09, 1986']

In [32]:
# create empty df to store fighters' tale of the tape
all_fighter_tott_df = pd.DataFrame(columns=config['fighter_tott_column_names'])

# organise fighter tale of the tape
fighter_tott_df = organise_fighter_tott(fighter_tott, config['fighter_tott_column_names'], url)

# show fighter tale of the tape
display(fighter_tott_df)

,FIGHTER,HEIGHT,WEIGHT,REACH,STANCE,DOB,URL
0,Jose Aldo,"5' 7""",135 lbs.,"70""",Orthodox,"Sep 09, 1986",http://ufcstats.com/fighter-details/d0f3959b4a...
